# TopicGPT: LLM-based Topic Modeling

Implementation of TopicGPT (NAACL 2024) with modifications:
- Uses **local LM Studio API** (`mistralai/ministral-3-3b`)
- 3-stage pipeline: **Generation → Refinement → Assignment**
- Evaluates with **Coherence (C_v)**, **IRBO Diversity**, and **Topic Quality**
- Checkpointing support for resumable execution

In [25]:
import os
import re
import json
import time
import random
import pickle
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Configuration

In [26]:
LIST_SUBJECT = ["cs", "math", "physics"]
VERSION = "v1"

BASE_DIR = Path("../../../../data/preprocess")
RESULT_DIR = Path("../../../../results/topicGpt/modeling")
CHECKPOINT_DIR = Path("../../../../models/topicGpt")

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)
    (CHECKPOINT_DIR / subject).mkdir(parents=True, exist_ok=True)

LLM_API_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "mistralai/ministral-3-3b"
LLM_TEMPERATURE = 0.2
LLM_MAX_TOKENS = 39000

GENERATION_SAMPLE_SIZE = 500      
GENERATION_BATCH_SIZE = 5         
GENERATION_MAX_BATCHES = 100      
ASSIGNMENT_BATCH_SIZE = 5       

TOP_N_WORDS = 10
RBO_P = 0.9

print(f"Subjects: {LIST_SUBJECT}")
print(f"LLM: {LLM_MODEL} @ {LLM_API_URL}")
print(f"Results: {RESULT_DIR.resolve()}")

Subjects: ['cs', 'math', 'physics']
LLM: mistralai/ministral-3-3b @ http://localhost:1234/v1/chat/completions
Results: /home/nedo/Kuliah/TA/Program/results/topicGpt/modeling


## LLM API Helper

In [27]:
def call_llm(system_prompt: str, user_prompt: str, max_retries: int = 3) -> str:
    """Call LM Studio API with retry logic."""
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": LLM_TEMPERATURE,
        "max_tokens": LLM_MAX_TOKENS,
    }
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                LLM_API_URL,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
            
            # Handle both OpenAI-style and LM Studio response formats
            if "choices" in data:
                return data["choices"][0]["message"]["content"].strip()
            elif "content" in data:
                return data["content"].strip()
            elif "output" in data:
                return data["output"].strip()
            else:
                return str(data)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt+1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"  LLM call failed after {max_retries} attempts: {e}")
                return ""

# Test connection
test_resp = call_llm("You are a helpful assistant.", "Say 'OK' if you can read this.")
print(f"LLM connection test: {test_resp[:100]}")

  Retry 1/3 after 1s: HTTPConnectionPool(host='localhost', port=1234): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f34dfbc16e0>: Failed to establish a new connection: [Errno 111] Connection refused'))
  Retry 2/3 after 2s: HTTPConnectionPool(host='localhost', port=1234): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f34dfbc1480>: Failed to establish a new connection: [Errno 111] Connection refused'))
  LLM call failed after 3 attempts: HTTPConnectionPool(host='localhost', port=1234): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f34dfbcd370>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM connection test: 


## Checkpoint Utilities

In [28]:
def save_checkpoint(data, name: str, subject: str):
    """Save checkpoint to disk."""
    path = CHECKPOINT_DIR / subject / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint saved: {path}")

def load_checkpoint(name: str, subject: str):
    """Load checkpoint from disk, return None if not found."""
    path = CHECKPOINT_DIR / subject / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint loaded: {path}")
        return data
    return None

## Data Loading

In [29]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load preprocessed dataset."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    return df

all_data = {}
for subject in LIST_SUBJECT:
    df = load_dataset(subject)
    all_data[subject] = df
    print(f"{subject}: {len(df):,} documents loaded")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Sample text: {str(df['text'].iloc[0])[:120]}...")

cs: 165,756 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: fault detection using immune based systems and formal language algorithms this paper describes two approaches for fault ...
math: 157,085 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: supersymmetry and homotopy the homotopical information hidden in a supersymmetric structure is revealed by considering d...
physics: 146,311 documents loaded
  Columns: ['title', 'submitted_date', 'text', 'tag_text']
  Sample text: critical dynamics of gelation shear relaxation and dynamic density fluctuations are studied within a rouse model general...


---
## Stage 1: Topic Generation

Iteratively prompt the LLM with batches of documents. For each batch, the LLM sees existing topics and either identifies existing topics or proposes new ones.

In [30]:
GENERATION_SYSTEM_PROMPT = """You are an expert topic analyst. Your task is to identify generalizable topics from academic paper abstracts.

Rules:
- Each topic must be GENERAL and BROAD enough to cover multiple papers
- Topic labels must be concise (2-5 words)
- Each topic needs a short description
- Do NOT create overly specific topics tied to a single paper
- Each topic should represent a SINGLE concept, not a combination
- You MUST output your response strictly as a valid JSON object."""

GENERATION_USER_TEMPLATE = """Below are the current topics discovered so far:

[Current Topics]
{topics}

[Document]
{document}

[Instructions]
1. Read the document above.
2. Determine if it fits an existing topic from the list.
3. Respond ONLY with a valid JSON object. Do not include markdown formatting, explanations, or extra text.

If an existing topic fits well, use this JSON schema:
{{
    "is_new": false,
    "topic_id": 42
}}

If no existing topic fits and you must create a new one, use this JSON schema:
{{
    "is_new": true,
    "label": "Your Concise Label",
    "description": "A short, generalized description of the concept"
}}

Your response:"""

In [31]:
import time
def format_topics_for_prompt(topics: dict) -> str:
    if not topics:
        return "No topics exist yet. You must create a [NEW] topic."
    
    formatted_lines = []
    for tid, t_info in topics.items():
        label = t_info.get("label", "Unknown Label")
        desc = t_info.get("description", "No description provided.")
        formatted_lines.append(f"ID {tid} | {label}: {desc}")
        
    return "\n".join(formatted_lines)

def stratified_sample_by_year(df, min_per_year=100, pct=0.05):
    years = pd.to_datetime(df['submitted_date']).dt.year
    
    sampled_indices = []
    
    for year, group_indices in df.groupby(years).groups.items():
        n_total = len(group_indices)
        
        n_pct = int(n_total * pct)
        n_to_sample = max(min_per_year, n_pct)
        
        n_to_sample = min(n_total, n_to_sample)
        
        year_sample = df.loc[group_indices].sample(n=n_to_sample, random_state=42).index.tolist()
        sampled_indices.extend(year_sample)
        
    print(f"  Sampling complete. Total docs for topic generation: {len(sampled_indices)}")
    return sampled_indices

def parse_generation_response(response: str, topics: dict):
    """
    Parses a JSON response from the LLM.
    Returns (assigned_topic_id, new_topic_dict).
    """
    response = response.strip()
    
    response = re.sub(r"^```json\s*", "", response, flags=re.IGNORECASE)
    response = re.sub(r"\s*```$", "", response)
    
    try:
        data = json.loads(response)
        
        if data.get("is_new") is False:
            tid = data.get("topic_id")
            if isinstance(tid, int) and tid in topics:
                return tid, None
                
        elif data.get("is_new") is True:
            label = data.get("label", "").strip()
            desc = data.get("description", "").strip()
            if label and desc:
                return None, {"label": label, "description": desc}
                
    except json.JSONDecodeError:
        print(f"  [Warning] Failed to parse JSON: {response}")
        
    return None, None

def generate_topics(df, subject):
    """Stage 1: one-doc-at-a-time generation with doc->topic mapping."""
    checkpoint = load_checkpoint("generation", subject)
    if checkpoint is not None and "doc_map" in checkpoint:
        topics    = checkpoint["topics"]
        doc_map   = checkpoint["doc_map"]    # {doc_idx: topic_id}
        processed = checkpoint["processed"]  # set of doc_idx already done
        print(f"  Resumed: {len(topics)} topics, {len(processed)} docs processed")
    else:
        topics, doc_map, processed = {}, {}, set()

    sampled_indices = stratified_sample_by_year(df, min_per_year=100, pct=0.05)
    texts = df["text"].fillna("").tolist()
    next_id = max(topics.keys(), default=0) + 1

    todo = [i for i in sampled_indices if i not in processed]
    if not todo:
        print("  All docs already processed.")
        return topics, doc_map

    pbar = tqdm(todo, desc="Generating topics")
    for doc_idx in pbar:
        text = texts[doc_idx][:38000]
        topics_str = format_topics_for_prompt(topics)
        user_prompt = GENERATION_USER_TEMPLATE.format(
            topics=topics_str, document=text
        )
        response = call_llm(GENERATION_SYSTEM_PROMPT, user_prompt)
        
        tid, new_topic = parse_generation_response(response, topics)
        

        if new_topic:
            topics[next_id] = new_topic
            tid = next_id
            next_id += 1

        if tid is not None:
            doc_map[doc_idx] = tid

        processed.add(doc_idx)
        pbar.set_postfix({"topics": len(topics), "mapped": len(doc_map)})

        if len(processed) % 50 == 0:
            save_checkpoint(
                {"topics": topics, "doc_map": doc_map, "processed": processed},
                "generation", subject
            )

    save_checkpoint(
        {"topics": topics, "doc_map": doc_map, "processed": processed},
        "generation", subject
    )
    print(f"  Generated {len(topics)} topics, mapped {len(doc_map)} docs")
    return topics, doc_map

In [32]:
all_topics  = {}
all_doc_maps = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC GENERATION: {subject.upper()}")
    print(f"{'='*60}")
    topics, doc_map = generate_topics(all_data[subject], subject)
    all_topics[subject]   = topics
    all_doc_maps[subject] = doc_map
    print(f"\n  Topics for {subject}:")
    for tid, t in sorted(topics.items()):
        print(f"    [{tid}] {t['label']}: {t['description']}")


TOPIC GENERATION: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/generation.pkl
  Resumed: 621 topics, 8906 docs processed
  Sampling complete. Total docs for topic generation: 8906
  All docs already processed.

  Topics for cs:
    [1] Web Text Database Classification: Automated methods for categorizing and probing search-only text databases via query-based classification techniques
    [2] AI Story Understanding: Exploration of computational methods for interpreting narrative structures and semantic coherence in textual data, including historical shifts and agent-based approaches
    [3] Parallel Debugging Systems: Techniques and tools for diagnosing errors in concurrent/parallel computing environments, focusing on visualization, nondeterminism handling, and distributed debugging mechanisms
    [4] Noninvertibility Theory: Exploration of mathematical and cryptographic properties defining invertibility limits in functions, particularly focusing on novel definitions (e.g., st

In [33]:
from collections import Counter

def show_significant_topics(all_topics, all_doc_maps, min_docs=2):
    """
    Displays statistics for topics with more than 'min_docs' assignments.
    """
    for subject, topics in all_topics.items():
        print(f"\n{'='*65}")
        print(f"SIGNIFICANT TOPICS (Docs > {min_docs-1}): {subject.upper()}")
        print(f"{'='*65}")
        
        # Get counts for the current subject
        doc_map = all_doc_maps.get(subject, {})
        counts = Counter(doc_map.values())
        
        # Sort topics by frequency (highest first)
        sorted_tids = sorted(counts.items(), key=lambda item: item[1], reverse=True)
        
        total_mapped = sum(counts.values())
        print(f"{'ID':<5} | {'Count':<8} | {'Share %':<8} | {'Topic Label'}")
        print("-" * 65)
        
        found_any = False
        for tid, count in sorted_tids:
            if count >= min_docs:
                label = topics.get(tid, {}).get('label', 'Unknown')
                share = (count / total_mapped) * 100 if total_mapped > 0 else 0
                
                print(f"{tid:<5} | {count:<8} | {share:>6.1f}%  | {label}")
                found_any = True
        
        if not found_any:
            print(f"No topics found with more than {min_docs-1} documents.")
            
        print(f"\nUnique topics in {subject}: {len(topics)}")
        print(f"Significant topics: {len([c for c in counts.values() if c >= min_docs])}")

# Execution
show_significant_topics(all_topics, all_doc_maps, min_docs=10)


SIGNIFICANT TOPICS (Docs > 9): CS
ID    | Count    | Share %  | Topic Label
-----------------------------------------------------------------
14    | 1085     |   12.2%  | Repository Management Systems
21    | 498      |    5.6%  | Reproducibility Pitfalls
140   | 496      |    5.6%  | Periodic Numeration Systems
23    | 487      |    5.5%  | Dynamic Market Optimization
153   | 469      |    5.3%  | Time-Series Pattern Discovery
16    | 439      |    4.9%  | Polysemantic Feature Modeling
143   | 292      |    3.3%  | Context-Dependent Classification
15    | 259      |    2.9%  | Recursive Definition Systems
25    | 222      |    2.5%  | Variable Word Rate Modeling
20    | 220      |    2.5%  | Foundations of Mathematical Randomness
73    | 211      |    2.4%  | Regulatory Agency Independence
29    | 205      |    2.3%  | Multi-Channel Rate Distortion Optimization
103   | 179      |    2.0%  | Conservative Parallel Simulation Scalability
173   | 176      |    2.0%  | Epistemic Blame Th

---
## Stage 2: Topic Refinement

Merge near-duplicate or overlapping topics using the LLM.

In [34]:
REFINEMENT_SYSTEM_PROMPT = """You are an expert at organizing topic taxonomies. Your task is to merge topics that are near-duplicates, synonyms, or heavily overlapping.

Rules:
- Only merge topics that are truly redundant or nearly identical
- Keep the most general and descriptive label
- Return the merge operations in the specified format
- If no merges are needed, return "None" """

REFINEMENT_USER_TEMPLATE = """
You are given a list of topics. Identify groups that should be merged because they are near-duplicates or heavily overlapping.

[Topic List]
{topics}

[Task]
Detect topics representing the same concept and propose merges.

[Output Format]
Return ONLY valid JSON.

{{
  "merges": [
    {{
      "merge_ids": [id1, id2, ...],
      "kept_id": id,
      "label": "New merged topic label",
      "description": "Short description of the merged topic",
      "confidence": 0.0
    }}
  ]
}}

Rules:
- "merge_ids" must contain all topic IDs being merged
- "kept_id" must be one of the IDs inside merge_ids
- "confidence" must be between 0.0 and 1.0

If no merges are needed return:

{{
  "merges": []
}}

Return JSON only. Do not include explanations.
"""

In [35]:
import json

def parse_refinement_response(response: str, topics: dict, min_confidence: float = 0.0) -> list:
    """Parse merge operations from structured LLM JSON response."""
    
    if not response:
        return []

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        # fallback: extract JSON if wrapped in text
        start = response.find("{")
        end = response.rfind("}") + 1
        if start == -1 or end == -1:
            return []
        try:
            data = json.loads(response[start:end])
        except json.JSONDecodeError:
            return []

    merges = []

    for item in data.get("merges", []):
        merge_ids = item.get("merge_ids", [])
        kept_id = item.get("kept_id")
        confidence = float(item.get("confidence", 0.0))

        # validation
        if not merge_ids or kept_id not in merge_ids:
            continue

        if confidence < min_confidence:
            continue

        merges.append({
            "merge_ids": [int(x) for x in merge_ids],
            "kept_id": int(kept_id),
            "label": item.get("label", "").strip(),
            "description": item.get("description", "").strip(),
            "confidence": confidence
        })

    return merges


def refine_topics(topics: dict, subject: str, min_confidence: float = 0.7) -> dict:
    """Stage 2: Merge near-duplicate topics."""

    checkpoint = load_checkpoint("refinement", subject)
    if checkpoint is not None:
        if isinstance(checkpoint, dict) and "topics" in checkpoint:
            return checkpoint["topics"], checkpoint.get("old_to_new", {})
        return checkpoint, {}

    refined = dict(topics)

    topics_str = format_topics_for_prompt(refined)
    user_prompt = REFINEMENT_USER_TEMPLATE.format(topics=topics_str)

    response = call_llm(REFINEMENT_SYSTEM_PROMPT, user_prompt)

    merges = parse_refinement_response(response, refined, min_confidence=min_confidence)

    if not merges:
        print("  No merges needed")
    else:
        # Apply strongest merges first
        merges = sorted(merges, key=lambda x: x["confidence"], reverse=True)

        used_ids = set()

        print(f"  Applying {len(merges)} merge(s):")

        for m in merges:

            merge_ids = set(m["merge_ids"])
            kept_id = m["kept_id"]

            # Skip if topics already merged
            if merge_ids & used_ids:
                continue

            # Validate existence
            valid_ids = [i for i in merge_ids if i in refined]
            if len(valid_ids) < 2:
                continue

            print(
                f"    Merge {valid_ids} -> [{kept_id}] "
                f"{m['label']} (conf={m['confidence']:.2f})"
            )

            # Update kept topic
            if kept_id in refined:
                refined[kept_id] = {
                    "label": m["label"],
                    "description": m["description"]
                }

            # Remove merged topics
            for mid in valid_ids:
                if mid != kept_id and mid in refined:
                    del refined[mid]

            used_ids.update(valid_ids)

    # ---- Reindex topics sequentially ----

    reindexed = {}
    old_to_new = {}

    for new_id, (old_id, topic) in enumerate(sorted(refined.items()), start=1):
        reindexed[new_id] = topic
        old_to_new[old_id] = new_id

    save_checkpoint({"topics": reindexed, "old_to_new": old_to_new}, "refinement", subject)

    print(f"  Refined: {len(topics)} -> {len(reindexed)} topics")

    return reindexed, old_to_new

In [36]:
all_refined_topics = {}
all_refined_doc_maps = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC REFINEMENT: {subject.upper()}")
    print(f"{'='*60}")

    refined, old_to_new = refine_topics(all_topics[subject], subject)
    all_refined_topics[subject] = refined

    # Remap doc_map through merge operations
    raw_doc_map = all_doc_maps.get(subject, {})
    remapped = {doc_idx: old_to_new.get(tid, tid)
                for doc_idx, tid in raw_doc_map.items()
                if old_to_new.get(tid, tid) in refined}
    all_refined_doc_maps[subject] = remapped
    print(f"  doc_map: {len(raw_doc_map)} -> {len(remapped)} entries after remap")

    print(f"\n  Refined topics for {subject}:")
    for tid, t in sorted(refined.items()):
        print(f"    [{tid}] {t['label']}: {t['description']}")


TOPIC REFINEMENT: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/refinement.pkl
  doc_map: 8879 -> 8879 entries after remap

  Refined topics for cs:
    [1] Web Text Database Classification: Automated methods for categorizing and probing search-only text databases via query-based classification techniques
    [2] AI Story Understanding: Exploration of computational methods for interpreting narrative structures and semantic coherence in textual data, including historical shifts and agent-based approaches
    [3] Parallel Debugging and Extension Language Debugging Systems: Exploration of techniques for diagnosing errors in concurrent/parallel computing environments through visualization, nondeterminism handling, and distributed debugging mechanisms, with extensions to automated interactions across hardware/software layers.
    [4] Noninvertibility Theory: Exploration of mathematical and cryptographic properties defining invertibility limits in functions, particularly focusing o

In [37]:
show_significant_topics(all_refined_topics, all_refined_doc_maps, min_docs=2)


SIGNIFICANT TOPICS (Docs > 1): CS
ID    | Count    | Share %  | Topic Label
-----------------------------------------------------------------
14    | 1085     |   12.2%  | Repository Management Systems
20    | 498      |    5.6%  | Reproducibility Pitfalls
138   | 496      |    5.6%  | Periodic Numeration Systems
22    | 487      |    5.5%  | Dynamic Market Optimization
151   | 469      |    5.3%  | Time-Series Pattern Discovery
16    | 439      |    4.9%  | Polysemantic Feature Modeling and Ambiguity Handling in Logic Systems
141   | 292      |    3.3%  | Context-Dependent Classification
15    | 259      |    2.9%  | Recursive Definition Systems
17    | 255      |    2.9%  | Predictor-Based Parsing
24    | 222      |    2.5%  | Variable Word Rate Modeling
19    | 220      |    2.5%  | Foundations of Mathematical Randomness
71    | 211      |    2.4%  | Regulatory Agency Independence
27    | 205      |    2.3%  | Multi-Channel Rate Distortion Optimization
101   | 179      |    2.0%  |

---
## Stage 2.1: Topic Enrichment

Use the LLM to write a richer 2-3 sentence description per topic, informed by the actual documents assigned to it during generation.

In [38]:
ENRICHMENT_SYSTEM_PROMPT = """You are an expert academic topic analyst.
Given a topic name and representative paper abstracts, write a detailed 5-7 sentence description
capturing the topic's scope, key methods, and applications.

OUTPUT RULES:
1. Return ONLY valid JSON: {"enriched_description": "..."}
2. Use PLAIN TEXT only. No markdown, no bolding (**), and no bullet points (-).
3. If you use quotes inside the description, use 'single quotes' so the JSON doesn't break.
4. Keep the entire description on ONE SINGLE LINE. No newlines inside the JSON value."""

import json
import re

def clean_and_parse_json(response):
    text = re.sub(r"```json\s*|```", "", response).strip()
    
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1:
        return None
    
    json_str = text[start:end+1]
    
    json_str = json_str.replace('\n', ' ').replace('\r', '')
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            # Matches everything between "enriched_description": " and the final "
            match = re.search(r'"enriched_description":\s*"(.*)"', json_str)
            if match:
                content = match.group(1)
                return {"enriched_description": content}
        except:
            pass
    return None

def enrich_topics(refined_topics: dict, doc_map: dict, texts: list, subject: str, top_k: int = 8) -> dict:
    """Stage 2.5: enriched descriptions via LLM using assigned docs."""
    checkpoint = load_checkpoint("enrichment", subject)
    if checkpoint is not None:
        return checkpoint["enriched_topics"]

    from collections import defaultdict
    topic_docs = defaultdict(list)
    for doc_idx, tid in doc_map.items():
        if tid in refined_topics:
            topic_docs[tid].append(doc_idx)

    enriched = {}
    for tid, topic in tqdm(refined_topics.items(), desc=f"Enriching {subject}"):
        all_indices = topic_docs.get(tid, [])
        step = max(1, len(all_indices) // top_k)
        doc_indices = all_indices[::step][:top_k] # Takes a representative spread

        # print(f"All indiecies len {len(all_indices)}, after take represeitative : {len(doc_indices)}")
        snippets = [texts[i][:2000] for i in doc_indices if i < len(texts)]
        # print(f"Snippets count : {len(snippets)}\n")
        snippets_str = "\n\n".join(f"Abstract {i+1}:\n{s}" for i, s in enumerate(snippets)) or "(no docs assigned)"
        # user_prompt = (
        #     f"Topic: {topic['label']}\n"
        #     f"Current description: {topic['description']}\n\n"
        #     f"Representative abstracts:\n{snippets_str}\n\n"
        #     "Write an enriched description as JSON."
        # )
        user_prompt = (
    f"Topic Label: {topic['label']}\n"
    f"Initial Definition: {topic['description']}\n\n" # Help it stay on track
    f"New Evidence (Abstracts):\n{snippets_str}\n\n"
    "Task: Synthesize the Initial Definition with the New Evidence to create a "
    "comprehensive, technical description. If the abstracts provide more specific "
    "methods or applications than the initial definition, prioritize the abstracts."
 )

        response = call_llm(ENRICHMENT_SYSTEM_PROMPT, user_prompt)

        parsed_data = clean_and_parse_json(response)

        enriched_desc = topic.get("description", "No description available.")
        if parsed_data and "enriched_description" in parsed_data:
            enriched_desc = parsed_data["enriched_description"]
        else:
            print(f"  [Warning] Parse failed for Topic {tid}, using fallback.")

        enriched[tid] = {**topic, "enriched_description": enriched_desc}
        if len(enriched) % 50 == 0:
            save_checkpoint({"enriched_topics": enriched}, "enrichment", subject)
        # print(f"Enriched: {enriched[tid]}")

    save_checkpoint({"enriched_topics": enriched}, "enrichment", subject)
    return enriched

In [39]:
all_enriched_topics = {}
for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TOPIC ENRICHMENT: {subject.upper()}")
    print(f"{'='*60}")
    texts_subj = all_data[subject]["text"].fillna("").tolist()
    enriched = enrich_topics(
        all_refined_topics[subject],
        all_refined_doc_maps[subject],
        texts_subj, subject,
        top_k=20
    )
    all_enriched_topics[subject] = enriched
    print(f"  Enriched {len(enriched)} topics")


TOPIC ENRICHMENT: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/enrichment.pkl
  Enriched 616 topics

TOPIC ENRICHMENT: MATH
  Checkpoint loaded: ../../../../models/topicGpt/math/enrichment.pkl
  Enriched 610 topics

TOPIC ENRICHMENT: PHYSICS
  Checkpoint loaded: ../../../../models/topicGpt/physics/enrichment.pkl
  Enriched 504 topics


In [40]:
from collections import Counter

for subject in LIST_SUBJECT:
    checkpoint = load_checkpoint("enrichment", subject)

    # Extract the dictionary of topics directly using the key
    topics = checkpoint.get("enriched_topics", {})

    print(f"{'='*50}")
    print(f"SUBJECT: {subject.upper()}")
    print(f"Total Unique Topics: {len(topics)}")
    print(f"{'-'*50}")

    if not topics:
        print("No topics found in checkpoint!")
    else:
        first_tid = next(iter(topics))
        has_doc_count = "doc_count" in topics[first_tid]

        if has_doc_count:
            # Scenario A
            for tid, topic_data in topics.items():
                count = topic_data['doc_count']
                label = topic_data.get('label', 'Unknown')
                print(f"  [ID: {tid:>3}] {count:>5} docs | {label}")
                
        else:
            # Scenario B
            # Now 'subject' is correctly "cs", so it will find your doc_map!
            doc_map = all_refined_doc_maps.get(subject, {})
            topic_counts = Counter(doc_map.values())
            
            for tid, topic_data in topics.items():
                count = topic_counts.get(tid, 0)
                label = topic_data.get('label', 'Unknown')
                print(f"  [ID: {tid:>3}] {count:>5} docs | {label}")

  Checkpoint loaded: ../../../../models/topicGpt/cs/enrichment.pkl
SUBJECT: CS
Total Unique Topics: 616
--------------------------------------------------
  [ID:   1]     1 docs | Web Text Database Classification
  [ID:   2]     7 docs | AI Story Understanding
  [ID:   3]    22 docs | Parallel Debugging and Extension Language Debugging Systems
  [ID:   4]     7 docs | Noninvertibility Theory
  [ID:   5]    29 docs | Nonmonotonic Logic Encoding
  [ID:   6]     5 docs | Neuro-Fuzzy Control Systems
  [ID:   7]     2 docs | Novelty-Based Retrieval Evaluation
  [ID:   8]    11 docs | Type Class Extensions
  [ID:   9]     3 docs | Quantitative Probabilistic Logic Programming
  [ID:  10]    55 docs | Prosody-Based Segmentation
  [ID:  11]     6 docs | Dynamic Semantics Resolution
  [ID:  12]    25 docs | Repair-Based Speech Correction
  [ID:  13]    90 docs | Object-Oriented Music Systems
  [ID:  14]  1085 docs | Repository Management Systems
  [ID:  15]   259 docs | Recursive Definition Syst

---
## Stage 3: Topic Assignment (Sentence Transformers)

Assign every document to the closest enriched topic using cosine similarity over pre-computed `.mmap` embeddings.
A tuning loop evaluates all 6 available embedding models and selects the best by Topic Quality (harmonic mean of C_v coherence and IRBO diversity).

In [41]:
from pathlib import Path
import numpy as np

EMBEDDING_DIR = Path("../../../../embedding")

MODEL_HF_MAP = {
    "allenai_specter2":                           "allenai/specter2",
    "BAAI_bge_base_en_v1.5":                    "BAAI/bge-base-en-v1.5",
    "all_distilroberta_v1":                       "sentence-transformers/all-distilroberta-v1",
    "all_mpnet_base_v2":                          "sentence-transformers/all-mpnet-base-v2",
    "intfloat_e5_base_v2":                        "intfloat/e5-base-v2",
    "sentence_transformers_all_MiniLM_L6_v2":     "sentence-transformers/all-MiniLM-L6-v2",
}

import re

def list_embedding_models(subject: str):
    return sorted(re.sub(r'_v1$', '', f.stem) for f in (EMBEDDING_DIR / subject).glob("*.mmap"))

def load_doc_embeddings(subject: str, model_name: str) -> np.ndarray:
    meta_path = EMBEDDING_DIR / subject / f"{model_name}_meta_v1.npy"
    if not meta_path.exists():
        meta_path = EMBEDDING_DIR / subject / f"{model_name}_v1_meta.npy"
    meta  = np.load(meta_path, allow_pickle=True).item()
    n, d  = meta["n_samples"], meta["emb_dim"]
    return np.memmap(
        EMBEDDING_DIR / subject / f"{model_name}_v1.mmap",
        dtype="float32", mode="r", shape=(n, d)
    )


In [42]:
import torch
import numpy as np

class Specter2Wrapper:
    def __init__(self, device=None):
        from transformers import AutoTokenizer
        from adapters import AutoAdapterModel
        
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"    Loading Specter2 on {self.device}...")
        
        # Load base model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')
        self.model = AutoAdapterModel.from_pretrained('allenai/specter2_base')
        
        # Load the default proximity adapter (standard for retrieval and matching)
        self.model.load_adapter("allenai/specter2", source="hf", set_active=True)
        self.model.to(self.device)
        self.model.eval()

    def encode(self, texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True, **kwargs):
        all_embeddings = []
        batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
        
        with torch.no_grad():
            for batch in batches:
                inputs = self.tokenizer(
                    batch, 
                    padding=True, 
                    truncation=True, 
                    return_tensors="pt", 
                    max_length=512
                ).to(self.device)
                
                output = self.model(**inputs)
                
                # Specter2 uses the first token (CLS) for the document embedding
                embs = output.last_hidden_state[:, 0, :]
                
                if normalize_embeddings:
                    embs = torch.nn.functional.normalize(embs, p=2, dim=1)
                    
                all_embeddings.append(embs.cpu().numpy())
                
        return np.vstack(all_embeddings)

In [43]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
def assign_topics_centroid(doc_embs, doc_map, enriched_topics, threshold=0.5, batch_size=2000):
    """
    Computes topic centroids from doc_map and assigns all docs based on cosine similarity.
    """
    topic_ids = sorted(enriched_topics.keys())
    topic_centroids = []
    valid_ids = []

    # Create centroids from Stage 1 doc_map
    for tid in topic_ids:
        # doc_map is {doc_idx: topic_id}
        seed_indices = [idx for idx, t_id in doc_map.items() if t_id == tid]
        
        if seed_indices:
            # Average the embeddings of the documents the LLM assigned to this topic
            cluster_embs = doc_embs[seed_indices]
            centroid = np.mean(cluster_embs, axis=0)
            # Normalize for cosine similarity
            centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
            topic_centroids.append(centroid)
            valid_ids.append(tid)

    topic_centroids = np.array(topic_centroids)
    
    rows = []
    # Batch assign all documents in doc_embs
    for start in range(0, len(doc_embs), batch_size):
        end = min(start + batch_size, len(doc_embs))
        batch = doc_embs[start:end]
        
        # Normalize batch for dot-product similarity
        batch_norm = batch / (np.linalg.norm(batch, axis=1, keepdims=True) + 1e-8)
        sims = np.dot(batch_norm, topic_centroids.T)
        
        best_idx = sims.argmax(axis=1)
        scores = sims.max(axis=1)

        for i, (b_idx, score) in enumerate(zip(best_idx, scores)):
            if score >= threshold:
                tid = valid_ids[b_idx]
                label = enriched_topics[tid]["label"]
            else:
                tid = -1
                label = "Outlier / Unassigned"

            rows.append({
                "doc_idx": start + i,
                "topic_id": tid,
                "topic_label": label,
                "confidence": float(score)
            })

    return pd.DataFrame(rows)

def compute_topic_words_ctfidf(assignment_df: pd.DataFrame, texts: list, top_n: int = 10) -> dict:
    from sklearn.feature_extraction.text import TfidfVectorizer
    import pandas as pd

    topic_docs = []
    tids = []
    
    for tid, grp in assignment_df.groupby("topic_id"):
        if tid == -1: continue 
        
        combined_text = " ".join([texts[i] for i in grp["doc_idx"].tolist() if i < len(texts)])
        topic_docs.append(combined_text)
        tids.append(tid)

    if not topic_docs:
        return {}

    vec = TfidfVectorizer(
        stop_words="english", 
        max_features=10000,
        ngram_range=(1, 2), 
        max_df=0.8
    )
    
    tfidf_matrix = vec.fit_transform(topic_docs)
    feature_names = vec.get_feature_names_out()
    
    topic_words = {}
    for i, tid in enumerate(tids):
        row = tfidf_matrix.getrow(i).toarray().flatten()
        top_ids = row.argsort()[-top_n:][::-1]
        topic_words[tid] = [feature_names[idx] for idx in top_ids if row[idx] > 0]
    return topic_words


def compute_coherence_irbo(assignment_df: pd.DataFrame, texts: list, top_n: int = 10) -> dict:
    """Compute C_v coherence and IRBO diversity, return dict with both + TQ."""
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
    from itertools import combinations
    import numpy as np
    
    topic_words = compute_topic_words_ctfidf(assignment_df, texts, top_n)
    word_lists  = [v for v in topic_words.values() if v]
    
    tokenized   = [t.lower().split() for t in texts]

    gensim_word_lists = []
    for words in word_lists:
        topic_unigrams = []
        for w in words:
            topic_unigrams.extend(w.split()) 
            
        unique_unigrams = list(dict.fromkeys(topic_unigrams))[:top_n]
        gensim_word_lists.append(unique_unigrams)

    try:
        dct = Dictionary(tokenized)
        cm  = CoherenceModel(
            topics=gensim_word_lists,
            texts=tokenized, 
            dictionary=dct, 
            coherence="c_v", 
            processes=5
        )
        cv  = cm.get_coherence()
    except Exception as e:
        print(f"    Coherence error: {e}")
        cv = 0.0

    def rbo(l1, l2, p=0.9):
        score, weight, s1, s2 = 0.0, 1.0, set(), set()
        for d in range(1, min(len(l1), len(l2)) + 1):
            s1.add(l1[d-1]); s2.add(l2[d-1])
            score += weight * len(s1 & s2) / d
            weight *= p
        return 1 - score  

    pairs = list(combinations(word_lists, 2))
    irbo  = float(np.mean([rbo(a, b) for a, b in pairs])) if pairs else 0.0
    tq    = 2 * cv * irbo / (cv + irbo + 1e-8)
    
    return {"coherence": cv, "irbo": irbo, "topic_quality": tq}
import pandas as pd

def filter_and_reindex_topics(df_assign: pd.DataFrame, min_docs: int = 10) -> pd.DataFrame:
    
    df_filtered = df_assign[df_assign["topic_id"] != -1].copy()
    
    topic_counts = df_filtered["topic_id"].value_counts()
    valid_topics = topic_counts[topic_counts >= min_docs].index
    
    df_filtered = df_filtered[df_filtered["topic_id"].isin(valid_topics)].copy()
    
    unique_topics = sorted(df_filtered["topic_id"].unique())
    topic_mapping = {old_id: new_idx for new_idx, old_id in enumerate(unique_topics)}
    
    df_filtered["original_topic_id"] = df_filtered["topic_id"]
    df_filtered["topic_id"] = df_filtered["topic_id"].map(topic_mapping)
    
    return df_filtered

In [44]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
all_results = {}

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"TUNING: {subject.upper()}")
    print(f"{'='*60}")
    gen_ckpt = load_checkpoint("generation", subject)
    if not gen_ckpt or "doc_map" not in gen_ckpt:
        print(f"Skipping {subject}: No doc_map found in generation checkpoint.")
        continue
    
    doc_map = gen_ckpt["doc_map"] 
    texts_subj = all_data[subject]["text"].fillna("").tolist()
    models     = list_embedding_models(subject)
    best_score, best_res = -1, None

    for model_name in models:
        print(f"  Model: {model_name}")
        if model_name == "allenai_specter2":
            st_model = Specter2Wrapper()
        else:
            hf_name  = MODEL_HF_MAP.get(model_name, model_name)
            st_model = SentenceTransformer(hf_name)
        ckpt = load_checkpoint(f"adjust_assignment_{model_name}", subject)
        if ckpt:
            df_assign = ckpt["assignment_df"]
            metrics   = ckpt["metrics"]
        else:
            doc_embs  = load_doc_embeddings(subject, model_name)
            df_assign = assign_topics_centroid(
                doc_embs, 
                doc_map, 
                all_enriched_topics[subject]
            )
            df_assign = filter_and_reindex_topics(df_assign, min_docs=500)
            metrics   = compute_coherence_irbo(df_assign, texts_subj)
            save_checkpoint({"assignment_df": df_assign, "metrics": metrics},
                            f"adjust_assignment_{model_name}", subject)

        print(f"    C_v={metrics['coherence']:.4f}  IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}")
        if metrics["topic_quality"] > best_score:
            best_score = metrics["topic_quality"]
            best_res   = {"model": model_name, "df": df_assign, "metrics": metrics}

    all_results[subject] = best_res
    print(f"  Best: {best_res['model']} (TQ={best_score:.4f})")

for subject, res in all_results.items():
    df = res["df"].copy()
    df["subject"]    = subject
    df["best_model"] = res["model"]
    df["coherence"]  = res["metrics"]["coherence"]
    df["irbo"]       = res["metrics"]["irbo"]
    out = RESULT_DIR / subject / "adjust_topicgpt_assignments.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)
    print(f"Saved: {out}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Rows: {len(df)}")


TUNING: CS
  Checkpoint loaded: ../../../../models/topicGpt/cs/generation.pkl
  Model: BAAI_bge_base_en_v1.5
  Checkpoint saved: ../../../../models/topicGpt/cs/adjust_assignment_BAAI_bge_base_en_v1.5.pkl
    C_v=0.6458  IRBO=0.9260  TQ=0.7609
  Model: all_distilroberta_v1
  Checkpoint saved: ../../../../models/topicGpt/cs/adjust_assignment_all_distilroberta_v1.pkl
    C_v=0.6622  IRBO=0.9143  TQ=0.7681
  Model: all_mpnet_base_v2
  Checkpoint saved: ../../../../models/topicGpt/cs/adjust_assignment_all_mpnet_base_v2.pkl
    C_v=0.6714  IRBO=0.9435  TQ=0.7845
  Model: allenai_specter2
    Loading Specter2 on cuda...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


  Checkpoint saved: ../../../../models/topicGpt/cs/adjust_assignment_allenai_specter2.pkl
    C_v=0.6485  IRBO=0.9339  TQ=0.7655
  Model: intfloat_e5_base_v2
  Checkpoint saved: ../../../../models/topicGpt/cs/adjust_assignment_intfloat_e5_base_v2.pkl
    C_v=0.6294  IRBO=0.9119  TQ=0.7447
  Model: sentence_transformers_all_MiniLM_L6_v2
  Checkpoint saved: ../../../../models/topicGpt/cs/adjust_assignment_sentence_transformers_all_MiniLM_L6_v2.pkl
    C_v=0.6415  IRBO=0.9320  TQ=0.7599
  Best: all_mpnet_base_v2 (TQ=0.7845)

TUNING: MATH
  Checkpoint loaded: ../../../../models/topicGpt/math/generation.pkl
  Model: BAAI_bge_base_en_v1.5
  Checkpoint loaded: ../../../../models/topicGpt/math/adjust_assignment_BAAI_bge_base_en_v1.5.pkl
    C_v=0.6655  IRBO=0.9027  TQ=0.7662
  Model: all_distilroberta_v1
  Checkpoint loaded: ../../../../models/topicGpt/math/adjust_assignment_all_distilroberta_v1.pkl
    C_v=0.6734  IRBO=0.9188  TQ=0.7772
  Model: all_mpnet_base_v2
  Checkpoint loaded: ../../..

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


  Checkpoint loaded: ../../../../models/topicGpt/math/adjust_assignment_allenai_specter2.pkl
    C_v=0.6226  IRBO=0.8917  TQ=0.7332
  Model: intfloat_e5_base_v2
  Checkpoint loaded: ../../../../models/topicGpt/math/adjust_assignment_intfloat_e5_base_v2.pkl
    C_v=0.6343  IRBO=0.9180  TQ=0.7503
  Model: sentence_transformers_all_MiniLM_L6_v2
  Checkpoint loaded: ../../../../models/topicGpt/math/adjust_assignment_sentence_transformers_all_MiniLM_L6_v2.pkl
    C_v=0.6718  IRBO=0.9064  TQ=0.7717
  Best: all_mpnet_base_v2 (TQ=0.7876)

TUNING: PHYSICS
  Checkpoint loaded: ../../../../models/topicGpt/physics/generation.pkl
  Model: BAAI_bge_base_en_v1.5
  Checkpoint loaded: ../../../../models/topicGpt/physics/adjust_assignment_BAAI_bge_base_en_v1.5.pkl
    C_v=0.6793  IRBO=0.9238  TQ=0.7829
  Model: all_distilroberta_v1
  Checkpoint loaded: ../../../../models/topicGpt/physics/adjust_assignment_all_distilroberta_v1.pkl
    C_v=0.7081  IRBO=0.9171  TQ=0.7992
  Model: all_mpnet_base_v2
  Checkp

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


  Checkpoint loaded: ../../../../models/topicGpt/physics/adjust_assignment_allenai_specter2.pkl
    C_v=0.6532  IRBO=0.9194  TQ=0.7638
  Model: intfloat_e5_base_v2
  Checkpoint loaded: ../../../../models/topicGpt/physics/adjust_assignment_intfloat_e5_base_v2.pkl
    C_v=0.6567  IRBO=0.9274  TQ=0.7689
  Model: sentence_transformers_all_MiniLM_L6_v2
  Checkpoint loaded: ../../../../models/topicGpt/physics/adjust_assignment_sentence_transformers_all_MiniLM_L6_v2.pkl
    C_v=0.7120  IRBO=0.9029  TQ=0.7962
  Best: all_distilroberta_v1 (TQ=0.7992)
Saved: ../../../../results/topicGpt/modeling/cs/adjust_topicgpt_assignments.csv
  Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'coherence', 'irbo']
  Rows: 77058
Saved: ../../../../results/topicGpt/modeling/math/adjust_topicgpt_assignments.csv
  Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'coherence', 'irbo']
  Rows: 108170

In [45]:
def print_final_summary(all_results, all_enriched_topics):
    print(f"\n{'='*60}")
    print(f"{'SUBJECT':<12} | {'TOPICS':<8} | {'DOCS':<8} | {'BEST MODEL'}")
    print(f"{'-'*60}")
    
    for subject, res in all_results.items():
        df = res["df"]
        # Count unique topics (excluding outlier -1 if present)
        n_topics = len(all_enriched_topics[subject])
        n_docs = len(df)
        model = res["model"]
        
        print(f"{subject:<12} | {n_topics:<8} | {n_docs:<8} | {model}")

# Run the summary
print_final_summary(all_results, all_enriched_topics)


SUBJECT      | TOPICS   | DOCS     | BEST MODEL
------------------------------------------------------------
cs           | 616      | 77058    | all_mpnet_base_v2
math         | 610      | 108170   | all_mpnet_base_v2
physics      | 504      | 77327    | all_distilroberta_v1


In [46]:
def display_best_model_topic_words(all_results: dict, all_data: dict, top_n: int = 10):
    print("\n" + "="*80)
    print("TOPIC WORDS FOR THE BEST MODEL PER SUBJECT")
    print("="*80)

    for subject, res in all_results.items():
        if res is None:
            print(f"\n[!] No successful results found for {subject.upper()}")
            continue

        best_model = res["model"]
        best_df = res["df"]
        
        # Grab the original texts for this subject
        texts_subj = all_data[subject]["text"].fillna("").tolist()

        print(f"\n--- SUBJECT: {subject.upper()} | BEST MODEL: {best_model} ---")

        # Generate the topic words using your existing function
        topic_words = compute_topic_words_ctfidf(best_df, texts_subj, top_n=top_n)

        if not topic_words:
            print("  No valid topics found (perhaps filtering was too strict).")
            continue

        # Create a mapping from topic_id to topic_label from the dataframe for nice printing
        # (This grabs the first label it sees for each topic_id)
        id_to_label = dict(zip(best_df['topic_id'], best_df['topic_label']))

        # Print each topic ID, its descriptive label, and its top words
        for tid in sorted(topic_words.keys()):
            words = topic_words[tid]
            label = id_to_label.get(tid, "Unknown Label")
            words_str = ", ".join(words)
            
            print(f"  Topic {tid: <2} | {label[:40]: <40} | Words: {words_str}")

In [47]:
# ... (your existing loop and CSV saving code) ...

# Call the function to display the top 10 words for the best models
display_best_model_topic_words(all_results, all_data, top_n=10)


TOPIC WORDS FOR THE BEST MODEL PER SUBJECT

--- SUBJECT: CS | BEST MODEL: all_mpnet_base_v2 ---
  Topic 0  | AI Story Understanding                   | Words: dialogue, conversational, story, narrative, conversation, dialogues, conversations, stories, emotional, dialog
  Topic 1  | Parallel Debugging and Extension Languag | Words: concurrency, mpi, bugs, fault, debugging, rust, model checking, lock, checking, shared memory
  Topic 2  | Nonmonotonic Logic Encoding              | Words: logics, logic programs, logic programming, calculus, propositional, order logic, linear logic, prolog, modal logic, datalog
  Topic 3  | Prosody-Based Segmentation               | Words: speech, audio, lip, speaker, audio visual, facial, eeg, sign language, eye, acoustic
  Topic 4  | Repair-Based Speech Correction           | Words: speech, speaker, asr, speech recognition, audio, acoustic, speech enhancement, automatic speech, tts, voice
  Topic 5  | Object-Oriented Music Systems            | Words: vid

## Adjustment Stage

In [48]:
from collections import Counter
import copy

filtered_enriched_topics = {}
filtered_doc_maps = {}

MIN_DOCS = 2

for subject in LIST_SUBJECT:
    print(f"\n{'='*60}")
    print(f"FILTERING TOPICS: {subject.upper()}")
    print(f"{'='*60}")
    
    # 1. Get the current topics and doc map for the subject
    old_topics = all_enriched_topics.get(subject, {})
    old_doc_map = all_refined_doc_maps.get(subject, {})
    
    # 2. Count exactly how many docs are in each topic
    topic_counts = Counter(old_doc_map.values())
    
    new_topics = {}
    new_doc_map = {}
    old_to_new_id_map = {}
    
    # 3. Filter and Re-index Topics
    new_id_counter = 1
    
    # Sort old keys to ensure consistent ordering during re-indexing
    for old_tid in sorted(old_topics.keys()):
        count = topic_counts.get(old_tid, 0)
        
        # Only keep if strictly greater than 2
        if count >= MIN_DOCS:
            old_to_new_id_map[old_tid] = new_id_counter
            
            # Deep copy to avoid mutating your original dictionary
            topic_data = copy.deepcopy(old_topics[old_tid])
            
            # Update the doc_count inside the dictionary just to be perfectly accurate
            topic_data['doc_count'] = count 
            
            new_topics[new_id_counter] = topic_data
            new_id_counter += 1
            
    # 4. Re-index the Document Map
    for doc_idx, old_tid in old_doc_map.items():
        if old_tid in old_to_new_id_map:
            new_tid = old_to_new_id_map[old_tid]
            new_doc_map[doc_idx] = new_tid
            
    # 5. Store the clean results
    filtered_enriched_topics[subject] = new_topics
    filtered_doc_maps[subject] = new_doc_map
    
    dropped_topics = len(old_topics) - len(new_topics)
    print(f"  Original Topics : {len(old_topics)}")
    print(f"  Dropped Topics  : {dropped_topics} (had 1 or fewer docs)")
    print(f"  Remaining Topics: {len(new_topics)}")

# Optional: Replace the old variables with the clean ones so the rest of your pipeline uses them
all_enriched_topics = filtered_enriched_topics
all_refined_doc_maps = filtered_doc_maps


FILTERING TOPICS: CS
  Original Topics : 616
  Dropped Topics  : 387 (had 1 or fewer docs)
  Remaining Topics: 229

FILTERING TOPICS: MATH
  Original Topics : 610
  Dropped Topics  : 347 (had 1 or fewer docs)
  Remaining Topics: 263

FILTERING TOPICS: PHYSICS
  Original Topics : 504
  Dropped Topics  : 268 (had 1 or fewer docs)
  Remaining Topics: 236
